In [1]:

import sys, time, warnings, os
from pathlib import Path

def find_root(marker='Data/train.csv'):
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(f'Không thấy repo root (marker {marker}) từ {Path.cwd()}')

ROOT = find_root()
sys.path.append(str(ROOT / 'Modeling' / 'Code'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from statsmodels.tsa.seasonal import STL
from eval_protocol import time_split_per_kpi, evaluate_protocol
from preprocess import preprocess_all

df = pd.read_csv(ROOT / 'Data' / 'train.csv')
df.columns = ['timestamp', 'value', 'label', 'kpi']

ROBUST  = False    
MAX_GAP = 5
print('Repo root:', ROOT)
print('Số KPI trong train.csv:', df.kpi.nunique())

Repo root: C:\Projects\anomaly-detection-fundamentals
Số KPI trong train.csv: 26


In [2]:
def run_kpi_stl(gk):
    step = int(pd.Series(np.diff(np.sort(gk.timestamp.values))).mode().iloc[0])
    gk = time_split_per_kpi(gk, 0.6, 0.2)
    p = (preprocess_all(gk, max_gap_points=MAX_GAP, norm_method='robust')
         .sort_values('timestamp').reset_index(drop=True))
    y = p.label.values.astype(int); spl = p.split.values
    if y[spl == 'val'].sum() == 0 or y[spl == 'test'].sum() == 0:
        return None                                        # skip KPI thiếu nhãn ở val/test
    period = int(round(86400 / step))                      # 1440 cho 60s, 288 cho 300s
    min_len = 2 * period                                   # STL cần >= 2 chu kỳ
    J = max(period // 15, 1)                               # trend/low_pass jump → tăng tốc ~200x, AP không đổi

    real = p[p.value_filled.notna()]
    seg_len = real.groupby('segment').size()
    seg_ok = seg_len[seg_len >= min_len].index            # segment đủ dài cho STL
    p['stl_resid'] = np.nan
    for sid in seg_ok:                                     # phân rã từng segment (không bắc cầu qua gap)
        i = p.index[(p.segment == sid) & p.value_filled.notna()]
        r = STL(p.loc[i, 'value_norm'].values, period=period, robust=ROBUST,
                seasonal_jump=1, trend_jump=J, low_pass_jump=J).fit()
        p.loc[i, 'stl_resid'] = r.resid

    got = p.stl_resid.notna().values
    if got.sum() == 0:
        return None
    rtr = p.loc[(spl == 'train') & got, 'stl_resid']       # center/scale robust từ residual TRAIN
    med = rtr.median(); mad = np.median(np.abs(rtr - med)) * 1.4826
    mad = mad if mad > 1e-9 else 1e-9
    score = np.abs(p.stl_resid.values - med) / mad
    vm = (spl == 'val') & got; tm = (spl == 'test') & got
    if y[vm].sum() == 0 or y[tm].sum() == 0:
        return None                                        # STL bỏ hết anomaly val/test (segment ngắn)
    te = (spl == 'test')
    r = evaluate_protocol(y[vm], score[vm], y[tm], score[tm], step_s=step,
                          y_test_full=y[te], valid_test=got[te])   # fix TTD + P/R/F1 công bằng
    return dict(kpi=gk.kpi.iloc[0][:8], n_anom_test=int(y[tm].sum()), period=period,
                n_seg_stl=len(seg_ok), cov_test=round(float(tm.sum() / max((spl == 'test').sum(), 1)), 3),
                AP=round(r['AP_pw'], 3), AP_val=round(r['AP_val'], 3), ROC=round(r['ROC'], 3),
                PW_P=round(r['PW']['precision'], 3), PW_R=round(r['PW']['recall'], 3),
                PW_F1=round(r['PW']['fbeta'], 3), thr_pw=round(r['PW']['threshold'], 4),
                ttd_s=r['TTD']['ttd_sec_mean'],
                _score=score[tm], _ts=p.timestamp.values[tm], _y=y[tm])


In [3]:
rows, skipped, SCORES = [], [], []
t0 = time.time()
for kpi, gk in df.groupby('kpi'):
    try:
        r = run_kpi_stl(gk.copy())
    except Exception as e:
        r = None
        print('  lỗi', kpi[:8], type(e).__name__)
    if r is None:
        skipped.append(kpi[:8])
    else:
        SCORES.append(pd.DataFrame({'kpi': r['kpi'], 'timestamp': r.pop('_ts'),
                                    'y': r.pop('_y'), 'score': r.pop('_score')}))
        rows.append(r)
        print('done', r['kpi'], '| AP', r['AP'], '| PW_F1', r['PW_F1'],
              '| seg', r['n_seg_stl'], 'cov', r['cov_test'])
print()
print('Skipped (' + str(len(skipped)) + '):', skipped)
print('Tổng thời gian: ' + str(round(time.time() - t0)) + 's')

done 02e99bd4 | AP 0.468 | PW_F1 0.42 | seg 12 cov 0.94
done 07927a9a | AP 0.007 | PW_F1 0.0 | seg 3 cov 1.0
done 09513ae3 | AP 0.003 | PW_F1 0.0 | seg 10 cov 0.999
done 18fbb1d5 | AP 0.167 | PW_F1 0.004 | seg 8 cov 0.999
done 1c35dbf5 | AP 0.282 | PW_F1 0.334 | seg 10 cov 0.999
done 40e25005 | AP 0.167 | PW_F1 0.254 | seg 2 cov 1.0
done 71595dd7 | AP 0.119 | PW_F1 0.161 | seg 4 cov 1.0
done 7c189dd3 | AP 0.502 | PW_F1 0.587 | seg 3 cov 1.0
done 88cf3a77 | AP 0.29 | PW_F1 0.254 | seg 1 cov 1.0
done 8bef9af9 | AP 0.377 | PW_F1 0.366 | seg 3 cov 1.0
done 8c892e55 | AP 0.151 | PW_F1 0.02 | seg 4 cov 0.906
done 9ee58794 | AP 0.755 | PW_F1 0.47 | seg 1 cov 1.0
done a40b1df8 | AP 0.433 | PW_F1 0.456 | seg 3 cov 0.908
done affb01ca | AP 0.383 | PW_F1 0.412 | seg 3 cov 1.0
done c58bfcba | AP 0.002 | PW_F1 0.0 | seg 13 cov 0.999
done cff6d3c0 | AP 0.088 | PW_F1 0.147 | seg 5 cov 0.997
done da403e4e | AP 0.69 | PW_F1 0.109 | seg 8 cov 1.0
done e0770391 | AP 0.151 | PW_F1 0.021 | seg 4 cov 0.909


In [4]:
res = pd.DataFrame(rows).sort_values('AP', ascending=False).reset_index(drop=True)
n_all = df.kpi.nunique()
mAP = res['AP'].mean(); mPW = res['PW_F1'].mean(); mcov = res['cov_test'].mean()
print(f'MACRO AP    = {mAP:.3f}  (trên {len(res)}/{n_all} KPI)')
print(f'MACRO PW_F1 = {mPW:.3f} | MACRO ROC = {res["ROC"].mean():.3f} | MACRO AP_val = {res["AP_val"].mean():.3f}')
print(f'Coverage test trung bình = {mcov:.3f} (phần điểm test được STL chấm)')
res

MACRO AP    = 0.280  (trên 18/26 KPI)
MACRO PW_F1 = 0.223 | MACRO ROC = 0.732 | MACRO AP_val = 0.340
Coverage test trung bình = 0.981 (phần điểm test được STL chấm)


,kpi,n_anom_test,period,n_seg_stl,cov_test,AP,AP_val,ROC,PW_P,PW_R,PW_F1,thr_pw,ttd_s
0,9ee58794,761,1440,1,1.000,0.755,0.062,0.906,0.342,0.752,0.470,4.9801,150.000000
1,da403e4e,238,1440,8,1.000,0.690,0.427,0.956,0.058,0.924,0.109,1.6860,7.500000
2,7c189dd3,63,1440,3,1.000,0.502,0.444,0.855,0.931,0.429,0.587,9.5570,73.846154
3,02e99bd4,1282,1440,12,0.940,0.468,0.220,0.797,0.380,0.470,0.420,5.8406,37.500000
4,a40b1df8,83,1440,3,0.908,0.433,0.426,0.816,0.620,0.360,0.456,8.2658,67.500000
5,affb01ca,76,1440,3,1.000,0.383,0.422,0.782,0.808,0.276,0.412,11.5120,95.000000
6,8bef9af9,73,1440,3,1.000,0.377,0.450,0.819,0.850,0.233,0.366,10.3456,93.333333
7,88cf3a77,183,1440,1,1.000,0.290,0.655,0.775,0.262,0.246,0.254,6.0984,720.000000
8,1c35dbf5,2709,1440,10,0.999,0.282,0.572,0.706,0.282,0.409,0.334,2.0807,108.000000
9,40e25005,112,1440,2,1.000,0.167,0.172,0.709,0.773,0.152,0.254,5.8205,37.500000


In [5]:
out_path = ROOT / 'Modeling' / 'Stats' / 'STL' / 'stl_per_kpi_config.json'
res.to_json(out_path, orient='records', indent=1)
ART = ROOT / 'Modeling' / 'Artifacts'; ART.mkdir(parents=True, exist_ok=True)
pd.concat(SCORES, ignore_index=True).to_parquet(ART / 'scores_STL.parquet', index=False)
print('Đã lưu config per-KPI:', out_path, '| score ->', ART / 'scores_STL.parquet')

Đã lưu config per-KPI: C:\Projects\anomaly-detection-fundamentals\Modeling\Stats\STL\stl_per_kpi_config.json | score -> C:\Projects\anomaly-detection-fundamentals\Modeling\Artifacts\scores_STL.parquet
